
# 01. Carga e ingeniería de características

## Objetivo

Estudiar la primera etapa de consolidación del modelo predictor:

**Datos originales → características candidatas**

El dataset utilizado por el modelo contiene las variables climáticas,
los casos de dengue y los rezagos temporales.

En esta etapa se estudia:

- carga del dataset;
- ordenamiento temporal;
- identificación de variables;
- creación de características avanzadas;
- agregaciones móviles;
- interacciones;
- indicadores de estacionalidad;
- tratamiento de valores faltantes.

Esta etapa precede a la reducción dimensional.


## 1. Configuración de rutas

In [ ]:

from pathlib import Path
from datetime import datetime

RUTA_PROYECTO = Path(
    r"C:\Users\marco\Documentos\investigacion"
    r"\machine_learning_idalina\6_redes_neuronales"
)

RUTA_DATOS_RAW = RUTA_PROYECTO / "2_datos" / "1_raw"

RUTA_PROCESADOS = RUTA_PROYECTO / "2_datos" / "2_procesados"

RUTA_RESULTADOS = RUTA_PROCESADOS / "resultados_mlp"

RUTA_PROCESADOS.mkdir(parents=True, exist_ok=True)
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Proyecto:")
print(RUTA_PROYECTO)

print("\nDatos originales:")
print(RUTA_DATOS_RAW)

print("\nDatos procesados:")
print(RUTA_PROCESADOS)

print("\nResultados MLP:")
print(RUTA_RESULTADOS)


## 2. Importación de bibliotecas

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 3. Cargar el dataset original

In [ ]:

RUTA_EXCEL = (
    RUTA_DATOS_RAW /
    "2_meteo_epi_2021-2026_1_rezagos.xlsx"
)

df = pd.read_excel(
    RUTA_EXCEL,
    parse_dates=["fecha"]
)

df = df.sort_values("fecha").reset_index(drop=True)

print("Filas:", len(df))
print("Columnas:", len(df.columns))

df.head()



## 4. Identificación de las características

El dataset original contiene las variables climáticas y sus rezagos.
El script de consolidación parte de estas variables antes de realizar
la reducción dimensional.


In [ ]:

print(df.columns.tolist())


## 5. Ingeniería de características

In [ ]:

def crear_features_avanzadas(df):

    df_adv = df.copy()

    # -----------------------------------------
    # Agregaciones de casos de dengue
    # -----------------------------------------

    for window in [2, 4, 8]:

        df_adv[f"casos_media_{window}w"] = (
            df_adv["casos_dengue"]
            .rolling(window)
            .mean()
        )

        df_adv[f"casos_max_{window}w"] = (
            df_adv["casos_dengue"]
            .rolling(window)
            .max()
        )

        df_adv[f"casos_tendencia_{window}w"] = (
            df_adv["casos_dengue"]
            .diff(window)
        )

    # -----------------------------------------
    # Agregaciones climáticas
    # -----------------------------------------

    for var in ["temp", "prec", "hum_rel"]:

        for window in [4, 8, 12]:

            df_adv[f"{var}_media_{window}w"] = (
                df_adv[var]
                .rolling(window)
                .mean()
            )

            df_adv[f"{var}_max_{window}w"] = (
                df_adv[var]
                .rolling(window)
                .max()
            )

    # -----------------------------------------
    # Interacciones
    # -----------------------------------------

    df_adv["prec_temp_ratio"] = (
        df_adv["prec"] /
        (df_adv["temp"] + 0.1)
    )

    df_adv["hum_temp_interaction"] = (
        df_adv["hum_rel"] *
        df_adv["temp"]
    )

    df_adv["temp_range"] = (
        df_adv["temp_max"] -
        df_adv["temp_min"]
    )

    # -----------------------------------------
    # Estacionalidad
    # -----------------------------------------

    df_adv["mes"] = df_adv["fecha"].dt.month

    df_adv["semana_del_ano"] = (
        df_adv["fecha"]
        .dt.isocalendar()
        .week
        .astype(int)
    )

    return df_adv


## 6. Ejecutar la ingeniería de características

In [ ]:

df_adv = crear_features_avanzadas(df)

df_adv = df_adv.ffill()
df_adv = df_adv.fillna(0)

print("Dimensiones finales:", df_adv.shape)

df_adv.head()



## Preguntas de estudio

1. ¿Por qué no se deben volver a crear los lags si ya están en el dataset?
2. ¿Qué información aporta una media móvil?
3. ¿Qué diferencia existe entre un lag y una tendencia?
4. ¿Por qué introducir interacciones entre variables climáticas?
5. ¿Por qué el orden temporal es fundamental?
